# Experiment: Diffusion Fake vs Real Reviews

Standalone diffusion experiment for fake-review detection using the local dataset.

This notebook mirrors the config/testing expectations of `experiment_trust_fake_reviews`:

1. Phase A balanced sampling
2. Stratified train/test split (`test_size=0.25`)
3. Train baseline + diffusion-denoised classifier
4. Save artifacts and run metadata


In [1]:
import json
from pathlib import Path

import pandas as pd

# Resolve project root by locating this experiment folder.
cwd = Path.cwd().resolve()
project_root = None
for root in [cwd, *cwd.parents]:
    if (root / 'experiment_trust_fake_review_diffusion').exists():
        project_root = root
        break
if project_root is None:
    raise FileNotFoundError('Could not locate project root containing experiment_trust_fake_review_diffusion/.')

import sys
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from experiment_trust_fake_review_diffusion.diffusion_review_pipeline import (
    DiffusionReviewConfig,
    run_diffusion_review_experiment,
    score_review_texts,
)

project_root


PosixPath('/Users/lohzh/Desktop/cs3263-repo')

In [2]:
# Configuration (mirrors trust-fake-reviews notebook style overrides)
SAMPLE_SIZES = globals().get('RUN_SAMPLE_SIZES', {})
RUN_LIMITS = globals().get('RUN_LIMITS', {})

PHASE_A_TARGET_ROWS = int(SAMPLE_SIZES.get('phase_a_target_rows', 240))
TEST_SIZE = float(globals().get('RUN_TEST_SIZE', 0.25))
RANDOM_STATE = int(globals().get('RUN_RANDOM_STATE', 42))
INFERENCE_SAMPLES = int(RUN_LIMITS.get('inference_samples', 32))

DATASET_PATH = project_root / 'data/raw/fake-reviews-dataset/fake reviews dataset.csv'
ARTIFACTS_DIR = project_root / 'experiment_trust_fake_review_diffusion' / 'artifacts' / 'diffusion_fake_review'

print('Dataset path:', DATASET_PATH)
print('Artifacts dir:', ARTIFACTS_DIR)
print('PHASE_A_TARGET_ROWS =', PHASE_A_TARGET_ROWS)
print('TEST_SIZE =', TEST_SIZE)
print('RANDOM_STATE =', RANDOM_STATE)
print('INFERENCE_SAMPLES =', INFERENCE_SAMPLES)


Dataset path: /Users/lohzh/Desktop/cs3263-repo/data/raw/fake-reviews-dataset/fake reviews dataset.csv
Artifacts dir: /Users/lohzh/Desktop/cs3263-repo/experiment_trust_fake_review_diffusion/artifacts/diffusion_fake_review
PHASE_A_TARGET_ROWS = 240
TEST_SIZE = 0.25
RANDOM_STATE = 42
INFERENCE_SAMPLES = 32


In [3]:
# Training run
config = DiffusionReviewConfig(
    dataset_path=DATASET_PATH,
    artifacts_dir=ARTIFACTS_DIR,
    phase_a_target_rows=PHASE_A_TARGET_ROWS,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    inference_samples=INFERENCE_SAMPLES,
)

result = run_diffusion_review_experiment(config)
print('Run info path:', result['run_info_path'])
result


Run info path: /Users/lohzh/Desktop/cs3263-repo/experiment_trust_fake_review_diffusion/artifacts/diffusion_fake_review/run_info.json


{'schema_version': 'diffusion_fake_review_experiment/v1',
 'generated_at_utc': '2026-04-09T17:17:28.410319+00:00',
 'artifacts_dir': '/Users/lohzh/Desktop/cs3263-repo/experiment_trust_fake_review_diffusion/artifacts/diffusion_fake_review',
 'run_info_path': '/Users/lohzh/Desktop/cs3263-repo/experiment_trust_fake_review_diffusion/artifacts/diffusion_fake_review/run_info.json',
 'metrics': [{'model': 'baseline_logistic_clean',
   'auc': 0.8566666666666667,
   'brier': 0.1683553427007988,
   'log_loss': 0.49900545847260674,
   'accuracy': 0.7333333333333333,
   'precision': 0.7916666666666666,
   'recall': 0.6333333333333333,
   'f1': 0.7037037037037037},
  {'model': 'diffusion_denoised_logistic',
   'auc': 0.8288888888888889,
   'brier': 0.17856019156642036,
   'log_loss': 0.5195225370220878,
   'accuracy': 0.7333333333333333,
   'precision': 0.7692307692307693,
   'recall': 0.6666666666666666,
   'f1': 0.7142857142857143,
   'mean_prediction_std': 0.21023529075104677,
   'denoiser_mse_t

In [4]:
# Load metrics and run info
run_info_path = Path(result['run_info_path'])
run_info = json.loads(run_info_path.read_text(encoding='utf-8'))

metrics_df = pd.DataFrame(run_info['metrics'])
print('Run summary:')
print('phase_a_rows_loaded =', run_info['phase_a_rows_loaded'])
print('train_rows =', run_info['train_rows'])
print('test_rows =', run_info['test_rows'])

metrics_df


Run summary:
phase_a_rows_loaded = 240
train_rows = 180
test_rows = 60


,model,auc,brier,log_loss,accuracy,precision,recall,f1,mean_prediction_std,denoiser_mse_test
0,baseline_logistic_clean,0.856667,0.168355,0.499005,0.733333,0.791667,0.633333,0.703704,NaN,NaN
1,diffusion_denoised_logistic,0.828889,0.178560,0.519523,0.733333,0.769231,0.666667,0.714286,0.210235,0.861217


In [5]:
# Inspect top uncertain test predictions
predictions_path = Path(run_info['artifacts']['phase_a_test_predictions'])
preds_df = pd.read_csv(predictions_path)

uncertain = preds_df.sort_values('prediction_std_diffusion', ascending=False).head(10)
uncertain[['record_id', 'label_truth', 'p_real_diffusion', 'p_fake_diffusion', 'prediction_std_diffusion', 'text']]


,record_id,label_truth,p_real_diffusion,p_fake_diffusion,prediction_std_diffusion,text
19,fake_reviews_131,1,0.430783,0.569217,0.353440,I have friends who are SCCA corner workers and...
23,fake_reviews_209,1,0.530288,0.469712,0.336771,"<div id=""video-block-R3UOCM6EJJF7JV"" class=""a-..."
20,fake_reviews_103,0,0.286407,0.713593,0.334775,We purchased two sets of this set and they loo...
37,fake_reviews_133,1,0.564471,0.435529,0.329121,The ramp is too skinny and steep. The dogs won...
49,fake_reviews_87,0,0.701302,0.298698,0.317345,"If you like her other books, you'll love this ..."
9,fake_reviews_122,1,0.404124,0.595876,0.314083,Puppy loves this donkey. She's a bit aggressiv...
42,fake_reviews_115,0,0.590793,0.409207,0.313736,I was invested in the story. The characters we...
35,fake_reviews_147,1,0.538307,0.461693,0.312903,"Cute indoor toy for inclement weather. Hard, s..."
46,fake_reviews_106,0,0.260035,0.739965,0.312611,This along with the chargers and an AC adapter...
11,fake_reviews_37,0,0.342098,0.657902,0.308441,Beautiful love story with a happy ending. Gre...


In [6]:
# Optional: score custom text examples
sample_scores = score_review_texts(
    [
        'Excellent quality and accurate description. Works as expected.',
        'Buy now!!! Limited stock!!! Best product ever!!!',
    ],
    artifacts_dir=ARTIFACTS_DIR,
    inference_samples=INFERENCE_SAMPLES,
    random_state=RANDOM_STATE + 123,
)

pd.DataFrame(sample_scores)


,index,text,p_real,p_fake,prediction_std
0,0,Excellent quality and accurate description. Wo...,0.752253,0.247747,0.290557
1,1,Buy now!!! Limited stock!!! Best product ever!!!,0.715367,0.284633,0.311459


## Optional Long Run (~10 minutes)

Use this cell to train with a much larger Phase A sample and heavier diffusion settings.
It writes artifacts to a separate folder so the quick-run artifacts remain intact.


In [7]:
# Long-run tuning profile (edit this cell, then run the long-run training cell)
# If your previous run was only a few seconds, use a heavier profile.

# QUICK PROFILES:
# - moderate: better than default, usually a few minutes
# - aggressive: targets much longer runtime on most laptops/workstations
PROFILE = 'aggressive'  # 'moderate' or 'aggressive'

if PROFILE == 'moderate':
    RUN_LONG = {
        'phase_a_target_rows': 20000,   # 0 => full dataset
        'test_size': 0.25,
        'random_state': 42,
        'diffusion_steps': 120,
        'latent_dim': 320,
        'max_features': 80000,
        'min_df': 2,
        'denoiser_samples_per_row': 20,
        'classifier_samples_per_row': 10,
        'inference_samples': 96,
    }
else:
    RUN_LONG = {
        'phase_a_target_rows': 0,       # full dataset (40k+ rows)
        'test_size': 0.25,
        'random_state': 42,
        'diffusion_steps': 180,
        'latent_dim': 384,
        'max_features': 100000,
        'min_df': 2,
        'denoiser_samples_per_row': 32,
        'classifier_samples_per_row': 16,
        'inference_samples': 128,
    }

print('PROFILE =', PROFILE)
print('RUN_LONG =')
for k, v in RUN_LONG.items():
    print(' ', k, '=', v)


PROFILE = aggressive
RUN_LONG =
  phase_a_target_rows = 0
  test_size = 0.25
  random_state = 42
  diffusion_steps = 180
  latent_dim = 384
  max_features = 100000
  min_df = 2
  denoiser_samples_per_row = 32
  classifier_samples_per_row = 16
  inference_samples = 128


In [8]:
# Optional long-run training (~10 minutes, machine dependent)
# Goal: improve quality by increasing data volume + diffusion training footprint.

from pathlib import Path
import json
import pandas as pd

# You can override these at runtime before executing this cell.
LONG_RUN = globals().get('RUN_LONG', {})

# Try to use substantially more data than the default 240 rows.
# Set RUN_LONG['phase_a_target_rows']=0 to use the full dataset.
LONG_PHASE_A_TARGET_ROWS = int(LONG_RUN.get('phase_a_target_rows', 12000))
LONG_TEST_SIZE = float(LONG_RUN.get('test_size', 0.25))
LONG_RANDOM_STATE = int(LONG_RUN.get('random_state', 42))

# Heavier settings than default to increase effective training signal.
LONG_DIFFUSION_STEPS = int(LONG_RUN.get('diffusion_steps', 80))
LONG_LATENT_DIM = int(LONG_RUN.get('latent_dim', 256))
LONG_MAX_FEATURES = int(LONG_RUN.get('max_features', 50000))
LONG_MIN_DF = int(LONG_RUN.get('min_df', 3))
LONG_DENOISER_SAMPLES_PER_ROW = int(LONG_RUN.get('denoiser_samples_per_row', 12))
LONG_CLASSIFIER_SAMPLES_PER_ROW = int(LONG_RUN.get('classifier_samples_per_row', 6))
LONG_INFERENCE_SAMPLES = int(LONG_RUN.get('inference_samples', 64))

LONG_ARTIFACTS_DIR = project_root / 'experiment_trust_fake_review_diffusion' / 'artifacts' / 'diffusion_fake_review_long_run'

print('LONG_PHASE_A_TARGET_ROWS =', LONG_PHASE_A_TARGET_ROWS)
print('LONG_TEST_SIZE =', LONG_TEST_SIZE)
print('LONG_RANDOM_STATE =', LONG_RANDOM_STATE)
print('LONG_DIFFUSION_STEPS =', LONG_DIFFUSION_STEPS)
print('LONG_LATENT_DIM =', LONG_LATENT_DIM)
print('LONG_MAX_FEATURES =', LONG_MAX_FEATURES)
print('LONG_DENOISER_SAMPLES_PER_ROW =', LONG_DENOISER_SAMPLES_PER_ROW)
print('LONG_CLASSIFIER_SAMPLES_PER_ROW =', LONG_CLASSIFIER_SAMPLES_PER_ROW)
print('LONG_INFERENCE_SAMPLES =', LONG_INFERENCE_SAMPLES)
print('LONG_ARTIFACTS_DIR =', LONG_ARTIFACTS_DIR)

long_config = DiffusionReviewConfig(
    dataset_path=DATASET_PATH,
    artifacts_dir=LONG_ARTIFACTS_DIR,
    phase_a_target_rows=LONG_PHASE_A_TARGET_ROWS,
    test_size=LONG_TEST_SIZE,
    random_state=LONG_RANDOM_STATE,
    diffusion_steps=LONG_DIFFUSION_STEPS,
    latent_dim=LONG_LATENT_DIM,
    max_features=LONG_MAX_FEATURES,
    min_df=LONG_MIN_DF,
    denoiser_samples_per_row=LONG_DENOISER_SAMPLES_PER_ROW,
    classifier_samples_per_row=LONG_CLASSIFIER_SAMPLES_PER_ROW,
    inference_samples=LONG_INFERENCE_SAMPLES,
)

long_result = run_diffusion_review_experiment(long_config)
print('Long-run run_info_path:', long_result['run_info_path'])

long_run_info = json.loads(Path(long_result['run_info_path']).read_text(encoding='utf-8'))
long_metrics_df = pd.DataFrame(long_run_info['metrics'])
print('long phase_a_rows_loaded =', long_run_info['phase_a_rows_loaded'])
print('long train_rows =', long_run_info['train_rows'])
print('long test_rows =', long_run_info['test_rows'])

long_metrics_df


LONG_PHASE_A_TARGET_ROWS = 0
LONG_TEST_SIZE = 0.25
LONG_RANDOM_STATE = 42
LONG_DIFFUSION_STEPS = 180
LONG_LATENT_DIM = 384
LONG_MAX_FEATURES = 100000
LONG_DENOISER_SAMPLES_PER_ROW = 32
LONG_CLASSIFIER_SAMPLES_PER_ROW = 16
LONG_INFERENCE_SAMPLES = 128
LONG_ARTIFACTS_DIR = /Users/lohzh/Desktop/cs3263-repo/experiment_trust_fake_review_diffusion/artifacts/diffusion_fake_review_long_run
Long-run run_info_path: /Users/lohzh/Desktop/cs3263-repo/experiment_trust_fake_review_diffusion/artifacts/diffusion_fake_review_long_run/run_info.json
long phase_a_rows_loaded = 40432
long train_rows = 30324
long test_rows = 10108


,model,auc,brier,log_loss,accuracy,precision,recall,f1,mean_prediction_std,denoiser_mse_test
0,baseline_logistic_clean,0.971760,0.063786,0.213780,0.912643,0.903619,0.923823,0.913609,NaN,NaN
1,diffusion_denoised_logistic,0.968499,0.105074,0.370194,0.911159,0.888556,0.940245,0.913670,0.202659,0.670396
